In [ ]:
# markdown

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

# 1. Fetch the database URL from your terminal's environment
db_url = os.environ.get("DATABASE_URL")

# Clean the URL formatting for SQLAlchemy (it requires 'postgresql://')
if db_url and db_url.startswith("postgres://"):
    db_url = db_url.replace("postgres://", "postgresql://", 1)

# 2. Initialize the connection engine
engine = create_engine(db_url)

# 3. Query your ai_tasks table to verify the connection and see your queue
query = "SELECT * FROM ai_tasks;"
df = pd.read_sql(query, engine)

# 4. Preview the dataframe
print(f"Successfully loaded {len(df)} rows from the database!")
df.head()

Successfully loaded 20 rows from the database!


,id,task_type,payload,status,retry_count,created_at,priority
0,1,health_check,{'check': 'daily'},completed,0,2026-07-19 17:36:53.512662,0
1,2,health_check,{'check': 'daily'},completed,0,2026-07-19 17:49:35.785690,0
2,3,autonomous_audit,"{'action': 'daily_health_check', 'source': 'sc...",completed,0,2026-07-19 18:00:09.361606,0
3,4,autonomous_audit,"{'action': 'daily_health_check', 'source': 'sc...",completed,0,2026-07-19 18:01:54.289748,0
4,5,autonomous_audit,"{'action': 'daily_health_check', 'source': 'sc...",failed,1,2026-07-19 18:02:57.555024,0


In [3]:
import pandas as pd

# 1. Query the actual property sandbox
query = "SELECT * FROM leads_sandbox;"
df_properties = pd.read_sql(query, engine)

# 2. Clean the dataframe to only include rows with an actual address
df_clean = df_properties.dropna(subset=['address']).copy()

# 3. Analyze the distribution of your lead ratings
print("--- Lead Rating Distribution ---")
rating_counts = df_clean['lead_rating'].value_counts()
print(rating_counts)

# 4. Preview the valid leads alongside your scraped notes
print("\n--- Lead Notes Preview ---")
preview_df = df_clean[['address', 'lead_rating', 'last_notes']].head()
preview_df

--- Lead Rating Distribution ---
lead_rating
B    19
Name: count, dtype: int64

--- Lead Notes Preview ---


,address,lead_rating,last_notes
0,Discovery Run - Rossmoor Market,B,None
1,"141 Glenview Ln, Rochester, NY 14609",B,"Rossmoor Scrape - $289,900"
2,"946 Arnett Blvd, Rochester, NY 14619",B,"Rossmoor Scrape - $170,000"
3,"4 Lorimer St, Rochester, NY 14608",B,"Rossmoor Scrape - $130,000"
4,"275 Biscayne Dr, Rochester, NY 14612",B,"Rossmoor Scrape - $149,000"


In [5]:
from sqlalchemy import text

# Delete the erroneous New York records
delete_query = "DELETE FROM leads_sandbox WHERE address LIKE '%Rochester%';"
with engine.begin() as conn:
    conn.execute(text(delete_query))

print("Cleared bad Rochester data from the sandbox.")

Cleared bad Rochester data from the sandbox.


In [6]:
import json
from sqlalchemy import text

# Update the payload to target the newly renamed file
task_payload = json.dumps({"command": "python3 scraper/scrape_redfin.py"})

# Insert the task
insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{task_payload}');
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

print("Queued the newly renamed Redfin scraper task!")

Queued the newly renamed Redfin scraper task!


In [7]:
import json
from sqlalchemy import text

# Queue the enrichment script payload
task_payload = json.dumps({"command": "python3 scraper/enrich_script.py"})

insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{task_payload}');
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

print("Enrichment task successfully dropped in queue!")

Enrichment task successfully dropped in queue!


In [8]:
import pandas as pd

# Fetch all leads currently residing in the sandbox
df_final = pd.read_sql("SELECT * FROM leads_sandbox;", engine)

print(f"Total entries in the data pipeline: {len(df_final)}")
# View the newest additions alongside their list origin tags
df_final.dropna(subset=['address'])[['address', 'lead_rating', 'last_notes']].tail(10)

Total entries in the data pipeline: 10


,address,lead_rating,last_notes
0,Discovery Run - Rossmoor Market,B,None
1,"1617 Ptarmigan Dr Unit 1A, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $650,000"
2,"3125 Terra Granada Dr #3, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $765,000"
3,"2100 Golden Rain Rd #3, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $830,000"
4,"1348 Rockledge Ln #7, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $350,000"
5,"2925 Golden Rain Rd #14, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $395,000"
6,"2741 Golden Rain Rd #7, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $585,000"
7,"1860 Tice Creek Dr #1432, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $390,000"
8,"3033 Golden Rain Rd #6, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $598,000"
9,"1840 Tice Creek Dr #2345, Walnut Creek, CA 94595",B,"Rossmoor Scrape - $235,000"


In [9]:
from sqlalchemy import text

# Expand the schema to hold deep property details
alter_queries = """
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS beds NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS baths NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS sqft NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS property_type VARCHAR(255);
"""

with engine.begin() as conn:
    conn.execute(text(alter_queries))

print("Schema expanded for detailed property extraction!")

Schema expanded for detailed property extraction!


In [10]:
import json
from sqlalchemy import text

# Queue the newly refactored crawler
task_payload = json.dumps({"command": "python3 scraper/scrape_redfin.py"})

insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{task_payload}');
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

print("Deep Stealth Crawler successfully queued. Grab a coffee, this will take some time to run patients loops.")

Deep Stealth Crawler successfully queued. Grab a coffee, this will take some time to run patients loops.


In [11]:
import pandas as pd

# Fetch the raw data flowing from the crawler
df_qa = pd.read_sql("SELECT * FROM leads_sandbox ORDER BY id DESC LIMIT 20;", engine)

print("--- PIPELINE QA REPORT ---")
print(f"Total Rows Inspected: {len(df_qa)}")
print(f"Missing Addresses: {df_qa['address'].isna().sum()}")
print(f"Missing Prices: {df_qa['price'].isna().sum()}")
print(f"Missing Structural Data (Beds/Baths): {df_qa['beds'].isna().sum()}")

# Confirm data types are numeric rather than text strings
print("\n--- SCHEMA DATATYPES ---")
print(df_qa[['price', 'beds', 'baths', 'sqft']].dtypes)

df_qa[['address', 'price', 'beds', 'baths', 'sqft']].head()

--- PIPELINE QA REPORT ---
Total Rows Inspected: 10
Missing Addresses: 0


KeyError: 'price'

In [12]:
from sqlalchemy import text

alter_queries = """
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS price NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS beds NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS baths NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS sqft NUMERIC;
ALTER TABLE leads_sandbox ADD COLUMN IF NOT EXISTS property_type VARCHAR(255);
"""

with engine.begin() as conn:
    conn.execute(text(alter_queries))

print("Schema successfully expanded! Columns added.")

Schema successfully expanded! Columns added.


In [17]:
import pandas as pd

# Query the task queue to see if the worker has picked them up
df_tasks = pd.read_sql("SELECT id, task_type, status, created_at FROM ai_tasks ORDER BY created_at DESC LIMIT 5;", engine)

print("--- AI TASKS QUEUE STATUS ---")
print(df_tasks)

--- AI TASKS QUEUE STATUS ---
   id       task_type  status                 created_at
0  26  execute_script  failed 2026-07-19 21:29:54.810986
1  25  execute_script  failed 2026-07-19 21:18:03.334622
2  24  execute_script  failed 2026-07-19 21:12:59.934479
3  23  execute_script  failed 2026-07-19 21:00:41.851861
4  22  execute_script  failed 2026-07-19 20:54:23.502831


In [15]:
import pandas as pd

# Fetch the complete row for the most recent failed task
failed_task = pd.read_sql("SELECT * FROM ai_tasks WHERE status = 'failed' ORDER BY created_at DESC LIMIT 1;", engine)

# Print all columns to see the payload and any captured error messages
for col in failed_task.columns:
    print(f"{col.upper()}:")
    print(failed_task[col].iloc[0])
    print("-" * 40)

ID:
26
----------------------------------------
TASK_TYPE:
execute_script
----------------------------------------
PAYLOAD:
{'command': 'python3 scraper/scrape_redfin.py'}
----------------------------------------
STATUS:
failed
----------------------------------------
RETRY_COUNT:
0
----------------------------------------
CREATED_AT:
2026-07-19 21:29:54.810986
----------------------------------------
PRIORITY:
1
----------------------------------------


In [27]:
import pandas as pd

# Check the queue to see if the worker picks up the next execution
df_tasks = pd.read_sql("SELECT id, task_type, status, created_at FROM ai_tasks ORDER BY created_at DESC LIMIT 5;", engine)
df_tasks

,id,task_type,status,created_at
0,29,execute_script,pending,2026-07-19 21:49:08.063538
1,28,execute_script,in_progress,2026-07-19 21:49:04.553031
2,27,execute_script,failed,2026-07-19 21:42:29.384742
3,26,execute_script,failed,2026-07-19 21:29:54.810986
4,25,execute_script,failed,2026-07-19 21:18:03.334622


In [19]:
import json
from sqlalchemy import text

# Queue a brand new task payload for the rebuilt worker
new_task_payload = json.dumps({"command": "python3 scraper/scrape_redfin.py"})

insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{new_task_payload}');
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

print("Fresh task queued! Let's check the queue again in 10-15 seconds.")

Fresh task queued! Let's check the queue again in 10-15 seconds.


In [26]:
import json
from sqlalchemy import text

# Queue a fresh task for the patched cloud worker
test_payload = json.dumps({"command": "python3 scraper/scrape_redfin.py"})

insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{test_payload}');
"""

with engine.begin() as conn:
    conn.execute(text(insert_query))

In [28]:
import pandas as pd

# Pull the latest entries to see the newly structured columns filling up
df_new = pd.read_sql("SELECT address, price, beds, baths, sqft, last_notes FROM leads_sandbox ORDER BY id DESC LIMIT 5;", engine)
df_new

,address,price,beds,baths,sqft,last_notes
0,Discovery Run - Rossmoor Market,None,None,None,None,None
1,"1617 Ptarmigan Dr Unit 1A, Walnut Creek, CA 94595",None,None,None,None,"Rossmoor Scrape - $650,000"
2,"3125 Terra Granada Dr #3, Walnut Creek, CA 94595",None,None,None,None,"Rossmoor Scrape - $765,000"
3,"2100 Golden Rain Rd #3, Walnut Creek, CA 94595",None,None,None,None,"Rossmoor Scrape - $830,000"
4,"1348 Rockledge Ln #7, Walnut Creek, CA 94595",None,None,None,None,"Rossmoor Scrape - $350,000"


In [29]:
from sqlalchemy import text

# Update all active and queued tasks to 'cancelled'
cancel_query = """
    UPDATE ai_tasks 
    SET status = 'cancelled' 
    WHERE status IN ('pending', 'in_progress');
"""

with engine.begin() as conn:
    conn.execute(text(cancel_query))

print("Queue cleared. All scraping jobs are officially cancelled.")

Queue cleared. All scraping jobs are officially cancelled.


In [30]:
import json
import time
import pandas as pd
from sqlalchemy import text

# 1. Queue a fresh task for the crawler
new_task_payload = json.dumps({"command": "python3 scraper/scrape_redfin.py"})
insert_query = f"""
    INSERT INTO ai_tasks (task_type, status, priority, payload) 
    VALUES ('execute_script', 'pending', 1, '{new_task_payload}');
"""
with engine.begin() as conn:
    conn.execute(text(insert_query))

print("Crawler task queued! Waiting 30 seconds for the cloud worker to boot and pull the first property...")
time.sleep(30)

# 2. Check the task status
df_tasks = pd.read_sql("SELECT id, status FROM ai_tasks ORDER BY created_at DESC LIMIT 1;", engine)
print(f"Current Worker Status: {df_tasks['status'].iloc[0].upper()}")

# 3. Pull the incoming structural data
df_qa = pd.read_sql("""
    SELECT address, price, beds, baths, sqft 
    FROM leads_sandbox 
    ORDER BY id DESC LIMIT 5;
""", engine)

print("\n--- INCOMING PIPELINE DATA ---")
df_qa

Crawler task queued! Waiting 30 seconds for the cloud worker to boot and pull the first property...
Current Worker Status: PENDING

--- INCOMING PIPELINE DATA ---


,address,price,beds,baths,sqft
0,Discovery Run - Rossmoor Market,None,None,None,None
1,"1617 Ptarmigan Dr Unit 1A, Walnut Creek, CA 94595",None,None,None,None
2,"3125 Terra Granada Dr #3, Walnut Creek, CA 94595",None,None,None,None
3,"2100 Golden Rain Rd #3, Walnut Creek, CA 94595",None,None,None,None
4,"1348 Rockledge Ln #7, Walnut Creek, CA 94595",None,None,None,None


In [31]:
# 1. Purge the old 'failed' logs from the queue
with engine.begin() as conn:
    conn.execute(text("DELETE FROM ai_tasks WHERE status = 'failed';"))

# 2. Queue the full crawler
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO ai_tasks (task_type, status, priority, payload) 
        VALUES ('execute_script', 'pending', 1, '{"command": "python3 scraper/scrape_redfin.py"}');
    """))

print("v3.0.0 Crawler initiated. The scraper is now working through the zip code list.")

v3.0.0 Crawler initiated. The scraper is now working through the zip code list.


In [34]:
# Run this periodically over the next 10 minutes
df_check = pd.read_sql("SELECT address, price, beds, baths, sqft FROM leads_sandbox WHERE price IS NOT NULL LIMIT 10;", engine)
df_check

,address,price,beds,baths,sqft
